# 08c · 微调 ESM-2：用 LoRA 改语言模型本身

## 这一章在整个项目里的位置

```
00–07 ✅ → [08 深度学习] → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 08a PyTorch 入门：ESM 向量 + 小网络 ✅
              ├─ 08b 图神经网络 ✅
              └─ 08c 微调 ESM-2（LoRA）   ← 你在这里
```

## 背景：为什么需要这一章

08b 的结论是：想再往上走，得**改进向量本身**。08a 和 08b 里 ESM-2 一直是冻结的：它写好的"笔记"一个字不改，我们只训练读笔记的那个小网络。

这一章让 ESM-2 本人也学：把我们的题目和答案拿给它，让它自己的参数跟着改。这就是**微调**（fine-tuning）。08a 里你问过"这算不算微调"，那时的答案是"不算"，这一章才是。

**用小模型**：你决定的。6.5 亿参数的 ESM-2 带着梯度训练，你的 16 GB Mac 装不下。我们用同一家族里 **3,500 万参数**的那个。代价是：结果不能直接和 08a 那个 6.5 亿的比，但我们会把差距量出来。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第一次运行会从 Hugging Face 下载小模型（约 130 MB）
   ├─ 训练在你的 Mac 的 GPU 上：这一章的批次大小固定，08b 的内存问题不会出现
   └─ 成绩记到 Databricks 的 MLflow，和前面三章记在同一个本子里
看结果：这里的输出；Databricks 网页的 Experiments
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 微调和 LoRA 是什么 | 在每页笔记旁边贴一张便利贴 |
| 2 | 为什么是小模型 | 量体裁衣 |
| 3 | 把蛋白切成一样长的片段 | 每批都用同一个尺寸的盒子装 |
| 4 | 三种用法，同一张卷子 | 同一个学生：不复习、只做笔记、真正去学 |
| 5 | 看结果 | 学了有用，但学偏了 |
| 6 | 08 章小结：最终用哪个 | 收官 |

**花费：0**。**时间**：约 20 分钟，其中微调约 12 分钟。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import torch
from databricks.sdk import WorkspaceClient
from transformers.utils import logging as hf_logging
from qafi_platform import lakehouse, tracking
from qafi_platform.lakehouse import sql
from qafi_core import baseline, finetune
hf_logging.set_verbosity_error()   # Hugging Face 的加载提示太多，只留错误
hf_logging.disable_progress_bar()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
MODELS = ROOT / "data/models"
MODELS.mkdir(parents=True, exist_ok=True)
print("Databricks:", w.config.host)
print("训练用的处理器:", finetune.device(), "| 模型:", finetune.MODEL, "@", finetune.REVISION[:8])

---

## 第 1 步 · 微调和 LoRA 是什么

### 在干什么
弄清楚"让大模型自己学"具体改的是什么，以及为什么要用 LoRA 这种省事的办法。

### 背景
**微调**：08a 里打过比方，ESM-2 是一位写好了笔记的专家。冻结 = 另请一个新人学着读他的笔记，专家本人什么都没学到；微调 = 把题目和答案拿给专家本人，让他自己改进，以后写出来的笔记就不一样了。技术上就是：训练的那四步（猜 → 对答案 → 找方向 → 改一点），"改一点"的范围扩大到 ESM-2 自己的参数。

**问题是参数太多**。小模型也有 3,500 万个参数。全部去改：
- 每个参数都要存梯度、存优化器的状态，内存翻好几倍
- 我们只有 279 个训练蛋白，3,500 万个旋钮一起拧，很容易把它原来学会的东西拧坏

**LoRA**（Low-Rank Adaptation，低秩适配）的办法：**原来的参数一个都不动，在旁边贴一张"便利贴"，只训练便利贴。**

ESM-2 里大部分参数在一个个大矩阵里。拿其中一个 480 × 480 的矩阵举例：
- 它有 480 × 480 = **23 万**个数
- LoRA 在旁边加两个**瘦长**的矩阵：480 × 8 和 8 × 480，一共 **7,680** 个数
- 两个瘦矩阵相乘，正好又是一个 480 × 480 的矩阵，把它**加到**原矩阵上，就是修改后的效果
- 也就是说，用 3% 的数，描述了对整个矩阵的一个修改

中间那个 8 叫**秩**（rank）：便利贴有多"宽"。越宽能改得越多，要训练的数也越多。

### 用了什么技术，为什么
- **LoRA**：现在微调大模型最常用的办法，大语言模型（比如给 ChatGPT 这类模型做专门领域的定制）也几乎都用它。好处：省内存；原模型不动，不怕改坏；一个原模型可以配好几张不同的便利贴，切换很方便
- **Hugging Face PEFT**：做 LoRA 的标准库（PEFT = parameter-efficient fine-tuning，"省参数的微调"）。一句话就能给模型贴上便利贴
- 我们给 ESM-2 每一层的注意力矩阵和前馈矩阵都贴了便利贴，秩 8。要训练的数是 **83 万**，占全部的 **2.4%**

### 面试怎么说
> "LoRA freezes the pretrained weights and learns a low-rank update next to each weight matrix: two thin matrices of rank r whose product is added to the original. With rank 8 on the attention and feed-forward layers of a 35M-parameter ESM-2, only 2.4% of the parameters are trained."

---

## 第 2 步 · 为什么是小模型

### 在干什么
说清楚这一章用的模型，以及这个选择的代价。

### 背景
ESM-2 是一个家族，从 800 万参数到 150 亿参数都有。我在你的 Mac 上实测了两个（每批 8 段、每段 256 个氨基酸，带 LoRA 训练）：

| 模型 | 每步耗时 | 占用内存 |
|---|---|---|
| ESM-2 3,500 万参数 | 0.5 秒 | 3.8 GB |
| ESM-2 1.5 亿参数 | 1.6 秒 | 9.6 GB |
| ESM-2 6.5 亿参数（08a 用的） | 没测：带梯度训练装不下 | |

1.5 亿那个要 9.6 GB，加上你平时开着的软件，16 GB 的 Mac 会开始大量用交换区，08b 已经出过一次这种事。所以用 3,500 万的。

**代价**：小模型本身就比大模型弱。所以这一章最公平的比较是**同一个小模型的三种用法**之间比，而不是拿小模型去和 08a 的大模型比。和大模型的差距我们也量一下，但要知道那不全是微调的功劳或问题。

### 公司里什么情况怎么做
- 先用小模型把整套流程走通、确认有没有效果，再决定要不要花钱上大模型和 GPU。这一章正好是这个"先走通"的角色
- 模型版本照样写死（`REVISION`），和 07e 一样，结果才能重做

### 面试怎么说
> "I measured memory before choosing: LoRA training of the 150M ESM-2 took 9.6 GB on a 16 GB laptop, the 35M one 3.8 GB, so the experiment uses the 35M model and compares its frozen and fine-tuned versions with each other."

---

## 第 3 步 · 把蛋白切成一样长的片段

### 在干什么
把训练数据整理成微调需要的样子。

### 背景
08a、08b 用的向量是 07e 一次算好存起来的。微调不行：ESM-2 自己在变，每一步都要把序列重新读一遍。所以这次喂给它的是**序列**本身。

比喻：**每批都用同一个尺寸的盒子装。**
- 每组实验只测了蛋白的一小段（Domainome 平均 58 个残基）。整条蛋白读进去太浪费，所以只切出**测过的那一段，再加上两边的上下文**，凑成 254 个氨基酸，加上 ESM-2 的开头、结尾两个记号，正好 **256**
- 测得长的（MaveDB 有的测了一千多个残基），就切成几段互相重叠一半的片段，每个突变归到它**最居中**的那一段（和 07e 读长蛋白是同一个办法）
- 不足 256 的片段用"空白"补齐，并告诉模型哪些是空白
- 每批固定 **8 段**，最后一批不够就用重复的片段补齐，但补的那些权重是 0，不参与学习

**为什么非要一样长**：08b 的事故。Mac 的 GPU 会给每一种新的输入大小编译并缓存一份，大小一直变，内存就一直涨。这次每一批都是 8 × 256，永远是同一个形状，所以可以放心用 GPU。

**答案放在一张"格子表"里**：每段 256 个位置 × 20 种氨基酸。测过的突变，在它的位置和氨基酸那一格填上答案和权重；没测过的格子权重是 0。一段只装一组实验的数据，所以同一个格子不会有两个答案。

### 执行
取训练突变和这些蛋白的序列，切片段：

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'train'", MODELS / "train_variants.parquet")
train = pd.read_parquet(MODELS / "train_variants.parquet")
VERSION = sql(w, f"DESCRIBE HISTORY {CATALOG}.gold.fct_variant_features LIMIT 1")["version"][0]
lakehouse.download(w, f"""SELECT d.uniprot_id, d.sequence FROM {CATALOG}.gold.dim_protein d
                          JOIN {CATALOG}.gold.protein_split s USING (uniprot_id) WHERE s.split = 'train'""", MODELS / "train_sequences.parquet")
sequences = pd.read_parquet(MODELS / "train_sequences.parquet").set_index("uniprot_id")["sequence"]
tokenizer, esm = finetune.load()   # 第一次会从 Hugging Face 下载
items = finetune.crops(train, sequences, tokenizer)
print(f"{len(items)} 段，装下了 {sum(len(c['cells']) for c in items):,} 个突变（一共 {len(train):,} 个）")
print(f"ESM-2 小模型: {sum(p.numel() for p in esm.parameters()) / 1e6:.1f} M 个参数，{esm.config.num_hidden_layers} 层，每个残基 {esm.config.hidden_size} 个数")
crop = items[0]
print(f"第一段: {crop['dataset']} | tokens {tuple(crop['tokens'].shape)} | 格子表 {tuple(crop['target'].shape)} | 测过的格子 {int((crop['weight'] > 0).sum())} 个")

### 结果怎么看
- **570 段**，841,400 个突变**一个不少**都放进了格子表（有一个测试专门检查每个突变正好落在一个格子里）
- 小模型：**3,350 万**个参数（名字里的 35M 是约数），**12 层**，每个残基用 **480** 个数描述（08a 的大模型是 33 层、1,280 个数）
- 每段：256 个记号，格子表 256 × 20

### 公司里什么情况怎么做
- 微调时数据的"形状"要提前设计好：固定长度、补齐、标记哪些是真数据。这在 NVIDIA 的 GPU 上也能提速，因为固定形状可以提前优化
- 切片段有代价：模型只看得到 254 个氨基酸的上下文，看不到整条蛋白。这是用省内存换来的

### 面试怎么说
> "Fine-tuning needs sequences, not cached embeddings, so each assay's measured stretch is cropped to 254 residues with flanking context, padded to a fixed 256 tokens, eight crops per batch, with labels on a 256 by 20 grid and zero weight where nothing was measured. Fixed shapes also avoid the Apple-GPU caching problem."

---

## 第 4 步 · 三种用法，同一张卷子

### 在干什么
用同一个小模型、同一批片段、和前面完全相同的 5 折，比较三种用法。

### 背景
比喻：**同一个学生，三种准备考试的方式。**

| 运行 | 用法 | 学生 |
|---|---|---|
| `esm2_35m_zero_shot` | 零样本：不训练，直接用它自己的打分（07e 的算法） | 不复习，凭老底子考 |
| `esm2_35m_frozen` | 冻结：ESM-2 算一次存起来，只训练后面的头（08a 的做法） | 不改脑子，只整理一份笔记 |
| `esm2_35m_lora` | LoRA 微调：头和便利贴一起训练 | 真正去学，脑子跟着变 |

后两个的区别**只有一个**：ESM-2 自己的参数动不动。头的结构、片段、标签、折都一样。

**冻结的那个为什么快**：冻结的 ESM-2 不管读几遍，输出都一样，所以每段只读**一次**，存起来，之后只训练头。LoRA 不行，ESM-2 每一步都在变，每一步都得重新读一遍。

训练轮数是先在一折上画学习曲线选的：冻结 4 轮，LoRA 5 轮。

### 执行
约 12 分钟，几乎全花在 LoRA 上：

In [ ]:
REFERENCES = {name: tracking.read_scores(w, run) for name, run in
              [("zero_shot", "esm2_zero_shot"), ("baseline", "baseline"), ("mlp_standardised", "mlp_standardised")]}
predictions, scores = {}, {}
for name, how in finetune.RUNS.items():
    predictions[name] = finetune.cross_validate(train, items, esm, how)
    scores[name] = baseline.dataset_scores(predictions[name])
    same_model = {f"small_{k.removeprefix('esm2_35m_')}": v for k, v in scores.items() if k != name}   # 和同一个小模型的其他用法配对比较
    tracking.cv_run(w, EXPERIMENT, name, {**finetune.parameters(how), "feature_table_version": VERSION},
                    scores[name], references={**REFERENCES, **same_model}, chapter="08c")
    print(name, baseline.summarise(scores[name]), "\n")
print("存回平台的预测:", f"{tracking.save_predictions(w, predictions['esm2_35m_lora'], 'esm2_35m_lora'):,}", "行")

### 结果怎么看
三个运行各打印一行成绩和 MLflow 链接。冻结的那个只要十几秒，LoRA 要十分钟上下：**同样的数据，微调贵了几十倍**。

### 公司里什么情况怎么做
- 微调之前，先做好"冻结"这个对照。它便宜，而且是判断微调值不值的唯一标尺
- 每一折都从**原版** ESM-2 重新开始，不能接着上一折微调过的继续，否则下一折的考题就被上一折"学过"了

### 面试怎么说
> "Three uses of the same small model on the same folds: zero-shot, frozen with a trained head, and LoRA fine-tuning. Frozen and LoRA differ only in whether the backbone's weights change; each fold restarts from the original checkpoint."

---

## 第 5 步 · 看结果

### 执行

In [ ]:
runs = tracking.latest_runs(EXPERIMENT)
runs.loc[[*finetune.RUNS][::-1] + ["mlp_standardised", "esm2_zero_shot"], ["all", "domainome", "proteingym", "mavedb"]].round(3)

### 结果怎么看
| 运行 | 全部 | domainome | proteingym | mavedb（功能） |
|---|---|---|---|---|
| `esm2_35m_lora`（小模型，微调） | 0.599 | 0.626 | 0.585 | 0.343 |
| `esm2_35m_frozen`（小模型，冻结） | 0.581 | 0.614 | 0.597 | 0.377 |
| `esm2_35m_zero_shot`（小模型，零样本） | 0.476 | 0.500 | 0.538 | 0.350 |
| `mlp_standardised`（08a，大模型冻结） | 0.627 | 0.652 | 0.634 | 0.378 |
| `esm2_zero_shot`（大模型，零样本） | 0.455 | 0.461 | 0.468 | 0.398 |

最关键的一对：同一个小模型，**微调和冻结**一组一组比：

In [ ]:
tracking.comparison(runs, "esm2_35m_lora", "small_frozen")

### 结果怎么看
| 数据 | 微调更好的实验占 | 平均变化 | p 值 |
|---|---|---|---|
| domainome | 67% | +0.014 | < 0.001 |
| proteingym | 70% | +0.022 | 0.19，不显著 |
| mavedb（功能） | 35% | **−0.014** | 0.01 |

三个结论，一个比一个重要：

**1. 微调本身是管用的。** 同一个小模型，让它自己学，稳定性比只训练头高了 0.014，三分之二的 Domainome 实验变好。LoRA 这套办法在我们的数据上是有效的。

**2. 但它学偏了：功能反而变差。** 最可能的解释和 08a 是同一个道理，只是更进一步：
- 训练数据里八成多是稳定性实验
- 冻结时，ESM-2 原来的知识（哪些位置进化上动不得）原封不动地留在向量里，头只是学着去读
- 微调时，ESM-2 自己被往"稳定性"的方向掰，原来那部分知识可能被冲淡了一些，而功能恰恰依赖那部分知识
- 要说明的是：微调后的模型在功能上仍然比它**自己的零样本**好（配对比较平均高 0.05），所以它没有把原来的本事丢光，只是不如冻结时保留得多。我没有直接测量"丢了多少"
- 大模型在微调中丢掉原有本事的现象，有个名字叫**灾难性遗忘**（catastrophic forgetting）。我们这里远不到"灾难"，但方向一样

**3. 小模型再怎么微调，也不如大模型冻结。** 微调过的小模型 0.599，冻结的大模型 0.627，三类数据全输（和 `mlp_standardised` 配对比，Domainome −0.025，功能 −0.031，ProteinGym −0.045）。在我们这里，**模型大小比微调更重要**。

一个顺带的观察：小模型的**零样本**在 Domainome 上反而比大模型的零样本略高（0.500 对 0.461），在功能上差得多（0.350 对 0.398）。原因我没有深究，可能和它只看 254 个氨基酸的片段有关：Domainome 测的是单个结构域，片段正好把注意力集中在结构域上。

### 公司里什么情况怎么做
- **微调不是免费的午餐**。它更贵、更慢，还可能把模型原来的本事冲掉。要先证明"冻结 + 头"不够用，再上微调
- 训练数据偏向哪里，微调就把模型往哪里掰。数据的构成比方法更要紧，这一点在 08a、08b、08c 反复出现
- 想用微调又不想丢原有本事，常见的做法有：混入一部分原来的训练任务一起练、调小学习率、只微调最后几层。这些留作以后

### 面试怎么说
> "LoRA beat the frozen version of the same 35M model on stability, plus 0.014, but lost 0.014 on functional assays: with 86% stability data, fine-tuning bent the backbone towards stability and diluted the evolutionary signal function relies on. A fine-tuned 35M model also stayed below the frozen 650M model on every assay type, so scale mattered more than fine-tuning here."

---

## 第 6 步 · 08 章小结：最终用哪个

### 在干什么
最后看一眼微调过的小模型能不能帮上功能集成，然后给整个 08 章下结论。

### 执行

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.ml.cv_predictions WHERE run_name IN ('baseline', 'mlp_standardised')", MODELS / "earlier_predictions.parquet")
earlier = pd.read_parquet(MODELS / "earlier_predictions.parquet").pivot(index=baseline.KEY, columns="run_name", values="predicted").reset_index()
members = predictions["esm2_35m_lora"].rename(columns={"predicted": "lora"}).merge(earlier, on=baseline.KEY)
combos = {"08a 的集成": ["esm_delta", "baseline", "mlp_standardised"], "再加上微调的小模型": ["esm_delta", "baseline", "mlp_standardised", "lora"]}
combined = {name: baseline.dataset_scores(members.assign(predicted=baseline.rank_average(members, columns))) for name, columns in combos.items()}
short = lambda column: column.replace("spearman_median_", "").replace("spearman_median", "all")
display(pd.DataFrame({name: baseline.summarise(s) for name, s in combined.items()}).T.drop(columns="datasets").rename(columns=short).round(3))
pd.Series(baseline.compare(combined["再加上微调的小模型"], combined["08a 的集成"])).filter(like="mavedb")

### 结果怎么看
| | 全部 | domainome | proteingym | mavedb（功能） |
|---|---|---|---|---|
| 08a 的集成 | 0.592 | 0.617 | 0.613 | 0.432 |
| 再加上微调的小模型 | 0.622 | 0.646 | 0.630 | 0.441 |

功能这一栏从 0.432 到 0.441，配对比较平均只高 **0.003**，p = 0.03：统计上显著，但很小。

**要不要加？不加。** 加它意味着以后每次更新模型、每次给全蛋白组打分，都要多跑一遍微调和推理，换来 0.003。这笔账不划算。这是一个**产品决定**，不是统计决定：显著不等于值得。

### 整个 08 章做了什么，学到了什么

| 章 | 试了什么 | 结果 | 学到什么 |
|---|---|---|---|
| 08a | 冻结大模型 + 小网络 | 稳定性 0.65，**最好**；功能不行 | 大模型的向量 + 简单的头，是性价比最高的做法 |
| 08a | 三个模型按排名平均 | 功能 0.43，**最好** | 各懂一半的模型，合起来才行 |
| 08b | 图神经网络 | 没提高 | 向量里已经有结构信息（探针 0.86–0.91） |
| 08c | LoRA 微调小模型 | 稳定性略升、功能下降，仍不如冻结大模型 | 微调会被数据掰偏；模型大小更要紧 |

**最终交给 09 号的模型：**

| 想问什么 | 用哪个 | 交叉验证成绩 |
|---|---|---|
| 会不会不稳定 | `mlp_standardised`（冻结的 ESM-2 650M + 小网络） | 0.65 |
| 会不会失去功能 | `ensemble`（零样本 + 树模型 + 上面那个网络，按排名平均） | 0.43 |

**功能为什么一直上不去**，三章下来答案是同一个：有功能数据的训练蛋白只有 29 个，方法再换也绕不过去。下一步真正有用的，是**更多的功能数据**，而不是更复杂的模型。

### 公司里什么情况怎么做
- 一个方向做完，要能用一张表说清楚：试了什么、结果、学到什么、最后选什么。这张表就是给老板和同事看的
- "最复杂的模型赢"在真实项目里很少见。赢的通常是**数据好**加上**一个简单可靠的模型**

### 面试怎么说
> "Across three chapters, the best stability model was a small head on frozen ESM-2 650M embeddings, and the best functional model a rank average of zero-shot, a tree model and that head. A GNN and LoRA fine-tuning did not improve on them; the binding constraint on functional prediction is the 29 training proteins with functional data, not the architecture."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 08c LoRA fine-tuning of a small ESM-2 and the chapter 08 decision") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 LoRA | 原参数不动，贴一张秩为 8 的便利贴 | 只训练 2.4% 的参数 |
| 2 小模型 | 实测内存后选 3,500 万参数的 | 先用小模型走通流程 |
| 3 片段 | 256 个记号一段，8 段一批，答案放在格子表里 | 固定形状，GPU 才安全 |
| 4 三种用法 | 零样本、冻结、LoRA，同一张卷子 | 冻结是判断微调值不值的标尺 |
| 5 结果 | 微调：稳定性 +0.014，功能 −0.014 | 微调会被数据掰偏；大小比微调要紧 |
| 6 08 章结论 | 最终模型不变 | 瓶颈是功能数据，不是模型 |

**一句话**：LoRA 让小模型自己学，稳定性涨了一点，功能却被训练数据掰偏了；小模型微调后仍不如冻结的大模型。

## 面试题

1. **What is LoRA and why use it?**
   > "It freezes the pretrained weights and trains a low-rank update next to each weight matrix. It needs a fraction of the memory, cannot destroy the original weights, and one base model can carry several adapters."

2. **When would you fine-tune instead of training a head on frozen embeddings?**
   > "Only after the frozen baseline is in place and clearly insufficient, and when the training data covers the target task. In our case fine-tuning helped the majority task, stability, and hurt the minority one, function."

3. **What is catastrophic forgetting?**
   > "Fine-tuning that overwrites what the pretrained model knew. Mixing in the original task, lower learning rates or adapters like LoRA reduce it."

4. **A smaller model fine-tuned, or a larger one frozen?**
   > "Here the frozen 650M model beat the fine-tuned 35M on every assay type. Scale of the pretrained representation mattered more than adapting a smaller one."

5. **Your ensemble improved by 0.003 with p of 0.03. Ship it?**
   > "No. Significant is not the same as worth it: the extra model has to be retrained and run for every proteome prediction. A product decision weighs the gain against that cost."

**下一步 09 · 全蛋白组预测**：用定下来的两个模型，先在**锁定的考试蛋白**上考一次（第一次碰考卷），和旧 QAFI、AlphaMissense 等在同一批突变上比，然后注册模型，给 20,652 个蛋白的每一个可能突变打分。